<a href="https://colab.research.google.com/github/marinazakimi/ECAA08--Grupo-06/blob/main/etapa-02-grafos/15_Simulacao_de_Vazamentos_e_Desvio_Automatico.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Aula 15 — Simulação de vazamentos e desvio automático

**ECAA08 · Grupo 06 · Drone agrícola de pulverização**


## Objetivo

Integrar o diagnóstico lógico da Etapa 1 à atualização da rede. O evento injetado informa um **trecho conhecido pelo simulador**; os sensores globais de pressão e vazão não permitem localizar sozinhos a mangueira rompida. Baixa pressão com baixa vazão continua gerando a hipótese `possivel_vazamento_ou_falha_bomba`, sem declarar certeza causal.


## Sequência implementada

1. Reter a ocorrência da falha e emitir comando simulado de bomba desligada e válvulas de pulverização fechadas.
2. Marcar o trecho e seu ramal como indisponíveis no modelo. Para e05/e06, solicitar o isolamento das duas extremidades do filtro principal.
3. Calcular uma rota candidata por Dijkstra e verificar acesso a **todos os quatro bicos**.
4. Registrar separadamente a confirmação de isolamento, o diagnóstico e a disponibilidade da alternativa.
5. Manter a bomba bloqueada durante a falha retida. A simulação não implementa rearme nem abertura automática da reserva.

O desvio automático desta entrega é o **recálculo e a proposta de reconfiguração**. Uma rota alternativa não autoriza retomar a aplicação com vazamento ativo. A confirmação física de fechamento, a inspeção, a validação hidráulica e o rearme pertencem a uma etapa posterior de implementação.


## Integração lógica

As regras R1–R8 e os limiares da Aula 10 são reconstruídos no notebook, sem importar arquivos da Etapa 1. A fusão foi ampliada: emergência, GPS, bateria, vento, modo, altitude, comunicação e diagnóstico entram no permissivo final da bomba. No código original da Aula 10, o comando da bomba não dependia de todos esses bloqueios globais; a correção está documentada no relatório de análise.

$$P_{bomba}=P_{global}\land P_{voo}\land P_{nivel}\land\neg D_{critico}.$$

No tratamento do vazamento acrescenta-se $\neg F_{retida}$; como a ocorrência permanece retida, a saída fica falsa. Telemetria ausente, não finita, fora de faixa ou com qualidade ruim também bloqueia a autorização.


## Cenários e resultados esperados

| Cenário | Resultado topológico | Comando da bomba |
|---|---|---|
| Falha e06, rede básica | Sem rota até os bicos | Desligado |
| Falha e06, rede redundante | Rota candidata de 2,2 m até o bico A | Desligado, aguarda inspeção |
| Fechamento não confirmado | Alternativa calculada, isolamento pendente | Desligado |
| Falha e07 após perda do ramal principal | Bomba isolada do distribuidor | Desligado |
| Perda da seção A | Outras seções não suprem a seção A | Aplicação completa indisponível |


## Criticidade e tempo de cálculo

Remover cada arco e testar a alcançabilidade identifica **cortes para o par tanque–bico A**. Isso é diferente de afirmar que todos são pontes do grafo não dirigido ou que toda a planta tem conectividade 2. O filtro reserva não duplica bomba, distribuidor nem saída de cada bico.

Mede-se somente o tempo computacional da rotina no computador que executou o notebook. Esse número não inclui detecção, comunicação, movimento de válvulas nem dinâmica do fluido, e não prova atendimento a um prazo de controle em tempo real.

**Entregável:** simulador com estado de falha retido, diário de eventos, recálculo, diagnóstico herdado e testes de falhas simples, duplas e de telemetria.


## Implementação executável

Python 3.10 ou superior. Execute de cima para baixo. As definições estão incluídas para permitir abrir apenas este notebook no Jupyter ou Colab. Nenhuma conexão com equipamento real é realizada.

In [17]:
from collections import deque, Counter
from math import inf, isfinite, hypot
from heapq import heappush, heappop
from itertools import permutations
from copy import deepcopy


In [18]:
def tabela(linhas):
    if not linhas:
        return '(sem registros)'
    cols = list(linhas[0])
    larg = {c: max(len(c), *(len(str(r.get(c, ''))) for r in linhas)) for c in cols}
    fmt = lambda r: ' | '.join(str(r.get(c, '')).ljust(larg[c]) for c in cols)
    return '\n'.join([fmt(dict(zip(cols, cols))), '-+-'.join('-'*larg[c] for c in cols)] + [fmt(r) for r in linhas])


In [19]:
def matriz_texto(M, linhas, colunas):
    return tabela([{'no': r, **{c: ('inf' if M[i][j] == inf else round(M[i][j], 3))
                   for j, c in enumerate(colunas)}} for i, r in enumerate(linhas)])


In [20]:
class GrafoTubulacao:
    """Dígrafo simples; catálogo físico e disponibilidade operacional separados.

    Sensores são atributos; válvulas podem estar nas arestas. Não há suposição
    de uma válvula em cada trecho. Arestas paralelas são rejeitadas.
    """
    def __init__(self, vertices):
        self.vertices = list(vertices)
        if len(set(self.vertices)) != len(self.vertices):
            raise ValueError('Vértices duplicados')
        self.arestas = {}
        self.adj = {v: [] for v in self.vertices}

    def adicionar_tubulacao(self, id_, origem, destino, comprimento_m,
                           valvula=None, diametro_mm=8.0, modos=('voo',)):
        if origem not in self.adj or destino not in self.adj:
            raise KeyError('Extremidade não cadastrada')
        if origem == destino or id_ in self.arestas:
            raise ValueError('Laço ou ID repetido')
        if any(self.arestas[e]['destino'] == destino for e in self.adj[origem]):
            raise ValueError('Aresta paralela: use um multigrafo com IDs próprios')
        if not isfinite(comprimento_m) or comprimento_m < 0 or not isfinite(diametro_mm) or diametro_mm <= 0:
            raise ValueError('Peso finito não negativo e diâmetro positivo são exigidos')
        self.arestas[id_] = dict(origem=origem, destino=destino, peso=float(comprimento_m),
                                 valvula=valvula, diametro_mm=diametro_mm,
                                 modos=tuple(modos), disponivel=True)
        self.adj[origem].append(id_)

    def vizinhos(self, u, modo=None, bloqueados=()):
        if u not in self.adj:
            raise KeyError(u)
        if u in bloqueados:
            return
        for id_ in self.adj[u]:
            a = self.arestas[id_]
            if a['disponivel'] and (modo is None or modo in a['modos']) and a['destino'] not in bloqueados:
                yield a['destino'], a['peso'], id_

    def bloquear(self, id_):
        self.arestas[id_]['disponivel'] = False

    def matrizes(self, modo=None):
        n = len(self.vertices)
        idx = {v: i for i, v in enumerate(self.vertices)}
        ids = list(self.arestas)
        A = [[0]*n for _ in range(n)]
        W = [[0.0 if i == j else inf for j in range(n)] for i in range(n)]
        B = [[0]*len(ids) for _ in range(n)]
        for j, id_ in enumerate(ids):
            a = self.arestas[id_]; u, v = idx[a['origem']], idx[a['destino']]
            # B descreve o catálogo físico, inclusive trechos indisponíveis.
            B[u][j], B[v][j] = -1, 1
            if a['disponivel'] and (modo is None or modo in a['modos']):
                A[u][v], W[u][v] = 1, a['peso']
        return A, W, B, ids

    def graus(self, modo=None):
        A, _, _, _ = self.matrizes(modo)
        return [{'no': v, 'entrada': sum(r[i] for r in A), 'saida': sum(A[i])}
                for i, v in enumerate(self.vertices)]


In [21]:
def criar_rede(redundante=False):
    nos = ['EST_TQ_01', 'EST_PMP_01', 'EST_MAN_01', 'DRN_TQ_01',
           'DRN_FLT_01', 'DRN_PMP_01', 'DRN_MAN_01'] + ['DRN_BICO_'+s for s in 'ABCD']
    if redundante:
        nos.append('DRN_FLT_02')
    g = GrafoTubulacao(nos)
    g.adicionar_tubulacao('e01', nos[0], nos[1], 1.5, diametro_mm=20, modos=('mistura','abastecimento'))
    g.adicionar_tubulacao('e02', nos[1], nos[2], 1.0, diametro_mm=20, modos=('mistura','abastecimento'))
    g.adicionar_tubulacao('e03', nos[2], nos[0], 2.0, 'EST_XV_REC', 20, ('mistura',))
    g.adicionar_tubulacao('e04', nos[2], nos[3], 3.0, 'EST_XV_01', 15, ('abastecimento',))
    g.adicionar_tubulacao('e05', nos[3], nos[4], 0.4, 'DRN_XV_F1_IN')
    g.adicionar_tubulacao('e06', nos[4], nos[5], 0.3, 'DRN_XV_F1_OUT')
    g.adicionar_tubulacao('e07', nos[5], nos[6], 0.5)
    for i, s in enumerate('ABCD'):
        g.adicionar_tubulacao(f'e{8+i:02}', nos[6], 'DRN_BICO_'+s, 0.6+0.2*i, 'DRN_XV_01'+s)
    if redundante:
        g.adicionar_tubulacao('e12', 'DRN_TQ_01', 'DRN_FLT_02', 0.5, 'DRN_XV_F2_IN')
        g.adicionar_tubulacao('e13', 'DRN_FLT_02', 'DRN_PMP_01', 0.6, 'DRN_XV_F2_OUT')
    return g


In [22]:
def reconstruir(pred, destino):
    if destino not in pred:
        return []
    caminho, u = [], destino
    while u is not None:
        caminho.append(u); u = pred[u]
    return caminho[::-1]


In [23]:
def bfs(g, origem, destino, modo=None, bloqueados=()):
    if origem not in g.adj or destino not in g.adj:
        raise KeyError('Origem/destino inexistente')
    if origem in bloqueados or destino in bloqueados:
        return [], {}, {}
    pred, nivel, fila = {origem: None}, {origem: 0}, deque([origem])
    while fila:
        u = fila.popleft()
        for v, _, _ in g.vizinhos(u, modo, bloqueados):
            if v not in pred:
                pred[v], nivel[v] = u, nivel[u]+1
                fila.append(v)
    return reconstruir(pred, destino), pred, nivel


In [24]:
def dfs_caminhos(g, origem, destino, modo=None, bloqueados=()):
    """Enumera caminhos simples. Custo depende do número de caminhos, não é O(V+E)."""
    if origem not in g.adj or destino not in g.adj:
        raise KeyError('Origem/destino inexistente')
    if origem in bloqueados or destino in bloqueados:
        return []
    resultado = []
    def visitar(u, caminho):
        if u == destino:
            resultado.append(caminho[:]); return
        for v, _, _ in g.vizinhos(u, modo, bloqueados):
            if v not in caminho:
                visitar(v, caminho+[v])
    visitar(origem, [origem])
    return resultado


In [25]:
def dijkstra(g, origem, destino, modo=None, bloqueados=()):
    if origem not in g.adj or destino not in g.adj:
        raise KeyError('Origem/destino inexistente')
    if origem in bloqueados or destino in bloqueados:
        return inf, []
    dist, pred, fila = {origem: 0.0}, {origem: None}, [(0.0, origem)]
    while fila:
        du, u = heappop(fila)
        if du != dist[u]:
            continue
        if u == destino:
            return du, reconstruir(pred, destino)
        for v, peso, _ in g.vizinhos(u, modo, bloqueados):
            if not isfinite(peso) or peso < 0:
                raise ValueError('Dijkstra exige pesos finitos não negativos')
            candidato = du+peso
            if candidato < dist.get(v, inf):
                dist[v], pred[v] = candidato, u
                heappush(fila, (candidato, v))
    return inf, []


In [26]:
REGRAS = [
    ('R1', {'nivel_critico'}, 'falta_insumo'),
    ('R2', {'bomba_ligada','valvula_aberta','vazao_baixa'}, 'falha_pulverizacao'),
    ('R3', {'bomba_ligada','vazao_baixa','pressao_alta'}, 'possivel_obstrucao'),
    ('R4', {'possivel_obstrucao'}, 'interromper_pulverizacao'),
    ('R5', {'falta_insumo'}, 'interromper_pulverizacao'),
    ('R6', {'bateria_critica'}, 'interromper_missao'),
    ('R7', {'vento_alto'}, 'suspender_pulverizacao'),
    ('R8', {'bomba_ligada','vazao_baixa','pressao_baixa'}, 'possivel_vazamento_ou_falha_bomba'),
]


In [27]:
def telemetria_nominal():
    return dict(nivel_reservatorio=80.0, vazao=2.0, pressao=3.0, bateria=75.0,
                vento=3.0, bomba_ligada=True, valvula_aberta=True, gps_ok=True,
                e_stop=False, auto_mode=True, manual_mode=False, is_flying=True,
                alt_ok=True, comunicacao_ok=True)


In [28]:
def avaliar_etapa1(t):
    """Regras R1–R8 da Aula 10; fusão final ampliada nesta etapa.

    Snapshot estabilizado: não simula a dinâmica de partida da bomba.
    Dados ausentes/inválidos bloqueiam a autorização.
    """
    faixas = {'nivel_reservatorio': (0,100), 'vazao': (0,10), 'pressao': (0,6),
              'bateria': (0,100), 'vento': (0,25)}
    booleanas = ['bomba_ligada','valvula_aberta','gps_ok','e_stop','auto_mode',
                 'manual_mode','is_flying','alt_ok','comunicacao_ok']
    valido = all(isinstance(t.get(k), (int,float)) and not isinstance(t.get(k),bool)
                 and isfinite(t[k]) and lo <= t[k] <= hi for k,(lo,hi) in faixas.items())
    valido = valido and all(isinstance(t.get(k),bool) for k in booleanas)
    if not valido or not t.get('comunicacao_ok'):
        return dict(permite=False, fatos=['telemetria_invalida'], regras=[])
    f = set()
    for k in ['bomba_ligada','valvula_aberta']:
        if t[k]: f.add(k)
    conds = {'nivel_baixo':t['nivel_reservatorio']<20, 'nivel_critico':t['nivel_reservatorio']<5,
             'vazao_baixa':t['bomba_ligada'] and t['vazao']<0.5,
             'pressao_alta':t['pressao']>5,
             'pressao_baixa':t['bomba_ligada'] and t['pressao']<1,
             'bateria_baixa':t['bateria']<20, 'bateria_critica':t['bateria']<10,
             'vento_alto':t['vento']>8}
    f.update(k for k,v in conds.items() if v)
    trilha = []
    while True:
        novos = [(r,c) for r,p,c in REGRAS if p <= f and c not in f]
        if not novos: break
        for r,c in novos:
            f.add(c); trilha.append(r)
    criticos = {'falta_insumo','falha_pulverizacao','possivel_obstrucao',
                'possivel_vazamento_ou_falha_bomba','interromper_pulverizacao',
                'interromper_missao','suspender_pulverizacao'}
    global_ok = (t['gps_ok'] and t['bateria']>=20 and t['vento']<=8 and not t['e_stop']
                 and (t['auto_mode'] ^ t['manual_mode']))
    permite = (global_ok and t['is_flying'] and t['alt_ok'] and t['nivel_reservatorio']>3
                and not bool(f & criticos))
    return dict(permite=permite, fatos=sorted(f), regras=trilha)


In [29]:
class SistemaDesvioAutomatico:
    """Simula isolamento e proposta de rota; não envia comandos a hardware."""
    def __init__(self, grafo):
        self.g = deepcopy(grafo)
        self.falha_retida = False
        self.historico = []

    def tratar_vazamento(self, trecho, t, isolamento_confirmado=False):
        self.falha_retida = True
        a = self.g.arestas[trecho]
        # e05/e06 pertencem ao mesmo ramal: duas válvulas propostas o isolam.
        isolamento = ['e05','e06'] if trecho in ('e05','e06') else [trecho]
        for id_ in isolamento: self.g.bloquear(id_)
        custo, rota = dijkstra(self.g, 'DRN_TQ_01', 'DRN_BICO_A', 'voo')
        todas = all(bool(dijkstra(self.g, 'DRN_TQ_01', 'DRN_BICO_'+s, 'voo')[1]) for s in 'ABCD')
        estado = ('AGUARDAR_CONFIRMACAO_ISOLAMENTO' if not isolamento_confirmado else
                  'ROTA_CANDIDATA_AGUARDAR_INSPECAO' if todas else 'SEM_ROTA_COMPLETA')
        reg = dict(evento=len(self.historico)+1, trecho=trecho, isolamento=isolamento,
                   estado=estado, rota_candidata=rota, custo_m=custo,
                   atende_todos_bicos=todas, diagnostico=avaliar_etapa1(t),
                   comando_bomba=False, valvulas_pulverizacao_abertas=False,
                   falha_retida=self.falha_retida)
        self.historico.append(reg)
        return reg


## Experimento 1 — Diagnóstico, isolamento e alternativa

In [30]:
from time import perf_counter
t=telemetria_nominal(); t.update(vazao=0.2,pressao=0.8)
assert 'possivel_vazamento_ou_falha_bomba' in avaliar_etapa1(t)['fatos']
registros=[]
for redundante in [False,True]:
    sistema=SistemaDesvioAutomatico(criar_rede(redundante))
    inicio=perf_counter()
    r=sistema.tratar_vazamento('e06',t,isolamento_confirmado=True)
    tempo_ms=(perf_counter()-inicio)*1000
    registros.append({'reserva':redundante,'estado':r['estado'],'custo_m':r['custo_m'],
                      'bomba':r['comando_bomba'],'tempo_calculo_ms':round(tempo_ms,4)})
    assert not r['comando_bomba'] and r['falha_retida']
    assert r['atende_todos_bicos']==redundante
    if redundante: assert abs(r['custo_m']-2.2)<1e-9
print(tabela(registros))
pendente=SistemaDesvioAutomatico(criar_rede(True)).tratar_vazamento('e06',t)
assert pendente['estado']=='AGUARDAR_CONFIRMACAO_ISOLAMENTO'
dupla=sistema.tratar_vazamento('e07',t,True)
assert dupla['rota_candidata']==[] and not dupla['comando_bomba']
print('Histórico da rede redundante:',sistema.historico)


reserva | estado                           | custo_m | bomba | tempo_calculo_ms
--------+----------------------------------+---------+-------+-----------------
False   | SEM_ROTA_COMPLETA                | inf     | False | 0.0497          
True    | ROTA_CANDIDATA_AGUARDAR_INSPECAO | 2.2     | False | 0.0919          
Histórico da rede redundante: [{'evento': 1, 'trecho': 'e06', 'isolamento': ['e05', 'e06'], 'estado': 'ROTA_CANDIDATA_AGUARDAR_INSPECAO', 'rota_candidata': ['DRN_TQ_01', 'DRN_FLT_02', 'DRN_PMP_01', 'DRN_MAN_01', 'DRN_BICO_A'], 'custo_m': 2.2, 'atende_todos_bicos': True, 'diagnostico': {'permite': False, 'fatos': ['bomba_ligada', 'falha_pulverizacao', 'possivel_vazamento_ou_falha_bomba', 'pressao_baixa', 'valvula_aberta', 'vazao_baixa'], 'regras': ['R2', 'R8']}, 'comando_bomba': False, 'valvulas_pulverizacao_abertas': False, 'falha_retida': True}, {'evento': 2, 'trecho': 'e07', 'isolamento': ['e07'], 'estado': 'SEM_ROTA_COMPLETA', 'rota_candidata': [], 'custo_m': inf, 'ate

## Experimento 2 — Bloqueios globais e limites herdados

In [31]:
nominal=telemetria_nominal()
assert avaliar_etapa1(nominal)['permite']
falhas=[{'e_stop':True},{'gps_ok':False},{'bateria':8},{'vento':9},
        {'alt_ok':False},{'comunicacao_ok':False},{'auto_mode':False},
        {'nivel_reservatorio':4},{'pressao':5.2,'vazao':0.25},
        {'pressao':float('nan')},{'pressao':None}]
for falha in falhas:
    t={**nominal,**falha}
    assert not avaliar_etapa1(t)['permite'],falha
    print('BLOQUEADO:',falha)
ausente=nominal.copy(); del ausente['pressao']
assert not avaliar_etapa1(ausente)['permite']
assert avaliar_etapa1({**nominal,'bateria':20,'vento':8,'nivel_reservatorio':5})['permite']
assert not avaliar_etapa1({**nominal,'bateria':19.99})['permite']
print('Limites de fronteira e dados ausentes conferidos.')


BLOQUEADO: {'e_stop': True}
BLOQUEADO: {'gps_ok': False}
BLOQUEADO: {'bateria': 8}
BLOQUEADO: {'vento': 9}
BLOQUEADO: {'alt_ok': False}
BLOQUEADO: {'comunicacao_ok': False}
BLOQUEADO: {'auto_mode': False}
BLOQUEADO: {'nivel_reservatorio': 4}
BLOQUEADO: {'pressao': 5.2, 'vazao': 0.25}
BLOQUEADO: {'pressao': nan}
BLOQUEADO: {'pressao': None}
Limites de fronteira e dados ausentes conferidos.


## Experimento 3 — Cortes para o par tanque–bico A

In [32]:
criticos=[]
g=criar_rede(True)
for id_,a in g.arestas.items():
    if 'voo' not in a['modos']: continue
    copia=deepcopy(g); copia.bloquear(id_)
    if not dijkstra(copia,'DRN_TQ_01','DRN_BICO_A','voo')[1]: criticos.append(id_)
assert criticos==['e07','e08']
print('Cortes de uma aresta para tanque -> bico A:',criticos)
secao=deepcopy(g); secao.bloquear('e08')
assert dijkstra(secao,'DRN_TQ_01','DRN_BICO_B','voo')[1]
assert not dijkstra(secao,'DRN_TQ_01','DRN_BICO_A','voo')[1]
print('Bico B alcançável não compensa a perda da seção A.')


Cortes de uma aresta para tanque -> bico A: ['e07', 'e08']
Bico B alcançável não compensa a perda da seção A.
